## MODEL EVALUATION

In [28]:
# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)


# ============================================================================
#  LOAD ALL TRAINED MODELS
# ============================================================================
print("Loading trained models...")
 
model_files = [
    'models/logistic_regression_model.pkl',
    'models/decision_tree_model.pkl',
    'models/random_forest_model.pkl',
    'models/xgboost_tuned_model.pkl',
]

model_names = [
    'Logistic Regression',
    'Decision Tree',
    'Random Forest',
    'XGBoost',
]
 
models = {}
for model_file, model_name in zip(model_files, model_names):
    try:
        models[model_name] = joblib.load(model_file)
        print(f"  ✓ Loaded: {model_name}")
    except:
        print(f"  ✗ Could not load: {model_file}")
 
print(f"\n✓ Loaded {len(models)} models")

# ============================================================================
# EVALUATE ALL MODELS
# ============================================================================
print("="*80)
print("STEP 3: EVALUATING MODELS ON TEST SET")
print("="*80)
 
results = []
all_predictions = {}
 
for model_name, model in models.items():
    print(f"\nEvaluating: {model_name}...")

     # --- THE FIX: Adjust labels for XGBoost models ---
    is_xgboost = "XGBoost" in model_name
    current_y_test = y_test - 1 if is_xgboost else y_test
    
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)
    
    # Store predictions
    all_predictions[model_name] = {
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    # Calculate metrics
    accuracy = accuracy_score(current_y_test, y_pred)
    f1_macro = f1_score(current_y_test, y_pred, average='macro')
    f1_weighted = f1_score(current_y_test, y_pred, average='weighted')
    precision_macro = precision_score(current_y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(current_y_test, y_pred, average='macro', zero_division=0)
    
    # ROC-AUC (One-vs-Rest for multiclass)
    try:
        roc_auc = roc_auc_score(current_y_test, y_pred_proba, multi_class='ovr', average='macro')
    except:
        roc_auc = 0.0
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'F1-Score (macro)': f1_macro,
        'F1-Score (weighted)': f1_weighted,
        'Precision (macro)': precision_macro,
        'Recall (macro)': recall_macro,
        'ROC-AUC': roc_auc
    })
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score (macro): {f1_macro:.4f}")
    print(f"  F1-Score (weighted): {f1_weighted:.4f}")
 

Loading trained models...
  ✓ Loaded: Logistic Regression
  ✓ Loaded: Decision Tree
  ✓ Loaded: Random Forest
  ✓ Loaded: XGBoost

✓ Loaded 4 models
STEP 3: EVALUATING MODELS ON TEST SET

Evaluating: Logistic Regression...
  Accuracy: 0.6489
  F1-Score (macro): 0.3290
  F1-Score (weighted): 0.6762

Evaluating: Decision Tree...
  Accuracy: 0.6538
  F1-Score (macro): 0.4032
  F1-Score (weighted): 0.6981

Evaluating: Random Forest...
  Accuracy: 0.6569
  F1-Score (macro): 0.4242
  F1-Score (weighted): 0.7046

Evaluating: XGBoost...
  Accuracy: 0.8135
  F1-Score (macro): 0.4767
  F1-Score (weighted): 0.8036


In [29]:
# ============================================================================
# STEP 4: CREATE COMPARISON DATAFRAME
# ============================================================================
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
 
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('F1-Score (macro)', ascending=False)
 
print("\n" + comparison_df.to_string(index=False))
 
# Find best model
best_model_row = comparison_df.iloc[0]
best_model_name = best_model_row['Model']
best_f1_score = best_model_row['F1-Score (macro)']
 
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   F1-Score (macro): {best_f1_score:.4f}")


MODEL COMPARISON SUMMARY

              Model  Accuracy  F1-Score (macro)  F1-Score (weighted)  Precision (macro)  Recall (macro)  ROC-AUC
            XGBoost  0.813526          0.476673             0.803649           0.517634        0.461430 0.880139
      Random Forest  0.656934          0.424197             0.704581           0.401940        0.637632 0.849064
      Decision Tree  0.653845          0.403197             0.698094           0.384402        0.578164 0.803415
Logistic Regression  0.648857          0.329023             0.676158           0.327833        0.399439 0.717671

🏆 BEST MODEL: XGBoost
   F1-Score (macro): 0.4767


In [30]:
print("\n" + "="*80)
print("STEP 5: GENERATING DETAILED REPORTS")
print("="*80)
 
detailed_results = []
 
for model_name in models.keys():
    y_pred = all_predictions[model_name]['y_pred']
    
    # Get per-class metrics
    report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    
    for severity in ['1', '2', '3', '4']:
        if severity in report_dict:
            detailed_results.append({
                'Model': model_name,
                'Severity': int(severity),
                'Precision': report_dict[severity]['precision'],
                'Recall': report_dict[severity]['recall'],
                'F1-Score': report_dict[severity]['f1-score'],
                'Support': report_dict[severity]['support']
            })
 
detailed_df = pd.DataFrame(detailed_results)
detailed_df.to_csv('detailed_evaluation_results.csv', index=False)
print("✓ Saved: detailed_evaluation_results.csv")


STEP 5: GENERATING DETAILED REPORTS
✓ Saved: detailed_evaluation_results.csv


In [32]:
models = {
    'Logistic Regression': joblib.load('models/logistic_regression_model.pkl'),
    'Decision Tree': joblib.load('models/decision_tree_model.pkl'),
    'Random Forest': joblib.load('models/random_forest_model.pkl'),
    'XGBoost': joblib.load('models/xgboost_tuned_model.pkl')
}
 
# Helper function to get correct y_test for each model
def get_labels(model_name, y_test):
    """Handle XGBoost label offset (0-3 vs 1-4)"""
    is_xgboost = 'XGBoost' in model_name
    return y_test - 1 if is_xgboost else y_test, [0,1,2,3] if is_xgboost else [1,2,3,4]
 
# ============================================================================
# CONFUSION MATRICES (Top 3 Models)
# ============================================================================
print("Creating confusion matrices...")
 
top_models = ['Random Forest', 'Decision Tree', 'XGBoost']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
for idx, model_name in enumerate(top_models):
    model = models[model_name]
    y_true, labels = get_labels(model_name, y_test)
    
    # Predict and create confusion matrix
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', 
                xticklabels=['Sev 1', 'Sev 2', 'Sev 3', 'Sev 4'],
                yticklabels=['Sev 1', 'Sev 2', 'Sev 3', 'Sev 4'],
                ax=axes[idx], cbar_kws={'label': 'Rate'})
    
    axes[idx].set_title(f'{model_name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('Actual', fontsize=10)
 
plt.suptitle('Confusion Matrices: Top 3 Models (Normalized)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✓ Saved: confusion_matrices.png\n")
plt.close()
 
# ============================================================================
# ROC CURVES (Best Model: XGBoost Tuned)
# ============================================================================
print("Creating ROC curves for best model (XGBoost Tuned)...")
 
model = models['XGBoost']
y_true, labels = get_labels('XGBoost', y_test)
 
# Get prediction probabilities
y_proba = model.predict_proba(X_test_scaled)
 
# Create 2x2 subplot for each severity class
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
 
severity_names = ['Severity 1', 'Severity 2', 'Severity 3', 'Severity 4']
 
for idx, (severity_label, severity_name) in enumerate(zip(labels, severity_names)):
    ax = axes[idx]
    
    # One-vs-Rest: binary classification for this severity
    y_binary = (y_true == severity_label).astype(int)
    y_score = y_proba[:, idx]
    
    # Calculate ROC
    fpr, tpr, _ = roc_curve(y_binary, y_score)
    auc = roc_auc_score(y_binary, y_score)
    
    # Plot
    ax.plot(fpr, tpr, linewidth=2.5, color='#2ecc71', label=f'AUC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5, label='Random')
    
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
    ax.set_title(f'{severity_name} (One-vs-Rest)', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
 
plt.suptitle('ROC Curves: XGBoost (Tuned) - Best Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/roc_curves_best_model.png', dpi=300, bbox_inches='tight')
print("✓ Saved: roc_curves_best_model.png\n")
plt.close()
 
# ============================================================================
# PER-CLASS PERFORMANCE BREAKDOWN
# ============================================================================
print("Creating per-class performance analysis...")
 
from sklearn.metrics import classification_report
 
# Get detailed metrics for best model
model = models['XGBoost']
y_true, labels = get_labels('XGBoost', y_test)
y_pred = model.predict(X_test_scaled)
 
# Classification report
report = classification_report(y_true, y_pred, labels=labels, 
                               target_names=severity_names, 
                               output_dict=True)
 
# Extract metrics
metrics_df = pd.DataFrame({
    'Severity': severity_names,
    'Precision': [report[name]['precision'] for name in severity_names],
    'Recall': [report[name]['recall'] for name in severity_names],
    'F1-Score': [report[name]['f1-score'] for name in severity_names],
    'Support': [report[name]['support'] for name in severity_names]
})
 
print("\nPer-Class Performance (XGBoost Tuned):")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)
 
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(severity_names))
width = 0.25
 
bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision', color='#3498db')
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall', color='#2ecc71')
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#f39c12')
 
ax.set_xlabel('Severity Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance: XGBoost (Tuned)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(severity_names)
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
 
# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)
 
plt.tight_layout()
plt.savefig('results/figures/per_class_performance.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: per_class_performance.png")
plt.close()
 
# Save metrics to CSV
metrics_df.to_csv('results/reports/per_class_metrics.csv', index=False)
print("✓ Saved: per_class_metrics.csv")
 

 

Creating confusion matrices...
✓ Saved: confusion_matrices.png

Creating ROC curves for best model (XGBoost Tuned)...
✓ Saved: roc_curves_best_model.png

Creating per-class performance analysis...

Per-Class Performance (XGBoost Tuned):
  Severity  Precision   Recall  F1-Score  Support
Severity 1   0.233978 0.272189  0.251641    845.0
Severity 2   0.863158 0.913568  0.887648  69176.0
Severity 3   0.623938 0.519331  0.566849  16683.0
Severity 4   0.349462 0.140632  0.200555   2311.0

✓ Saved: per_class_performance.png
✓ Saved: per_class_metrics.csv


In [34]:
models = {
    'Logistic Regression': joblib.load('models/logistic_regression_model.pkl'),
    'Decision Tree': joblib.load('models/decision_tree_model.pkl'),
    'Random Forest': joblib.load('models/random_forest_model.pkl'),
    'XGBoost': joblib.load('models/xgboost_tuned_model.pkl')
}
 
# Helper function to get correct y_test for each model
def get_labels(model_name, y_test):
    """Handle XGBoost label offset (0-3 vs 1-4)"""
    is_xgboost = 'XGBoost' in model_name
    return y_test - 1 if is_xgboost else y_test, [0,1,2,3] if is_xgboost else [1,2,3,4]
 

 
# ============================================================================
# ROC CURVES (Best Model: XGBoost Tuned)
# ============================================================================
print("Creating ROC curves for best model (XGBoost Tuned)...")
 
model = models['Random Forest']  # Change to 'XGBoost' for best model
y_true, labels = get_labels('Random Forest', y_test)
 
# Get prediction probabilities
y_proba = model.predict_proba(X_test_scaled)
 
# Create 2x2 subplot for each severity class
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()
 
severity_names = ['Severity 1', 'Severity 2', 'Severity 3', 'Severity 4']
 
for idx, (severity_label, severity_name) in enumerate(zip(labels, severity_names)):
    ax = axes[idx]
    
    # One-vs-Rest: binary classification for this severity
    y_binary = (y_true == severity_label).astype(int)
    y_score = y_proba[:, idx]
    
    # Calculate ROC
    fpr, tpr, _ = roc_curve(y_binary, y_score)
    auc = roc_auc_score(y_binary, y_score)
    
    # Plot
    ax.plot(fpr, tpr, linewidth=2.5, color='#2ecc71', label=f'AUC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5, label='Random')
    
    ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
    ax.set_title(f'{severity_name} (One-vs-Rest)', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
 
plt.suptitle('ROC Curves: Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/rf_roc_curves_best_model.png', dpi=300, bbox_inches='tight')
print("✓ Saved: rf_roc_curves_best_model.png\n")
plt.close()
 
# ============================================================================
# PER-CLASS PERFORMANCE BREAKDOWN
# ============================================================================
print("Creating per-class performance analysis...")
 
from sklearn.metrics import classification_report
 
# Get detailed metrics for best model
model = models['Random Forest']  # Change to 'XGBoost' for best model
y_true, labels = get_labels('Random Forest', y_test)
y_pred = model.predict(X_test_scaled)
 
# Classification report
report = classification_report(y_true, y_pred, labels=labels, 
                               target_names=severity_names, 
                               output_dict=True)
 
# Extract metrics
metrics_df = pd.DataFrame({
    'Severity': severity_names,
    'Precision': [report[name]['precision'] for name in severity_names],
    'Recall': [report[name]['recall'] for name in severity_names],
    'F1-Score': [report[name]['f1-score'] for name in severity_names],
    'Support': [report[name]['support'] for name in severity_names]
})
 
print("\nPer-Class Performance (Random Forest):")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)
 
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(severity_names))
width = 0.25
 
bars1 = ax.bar(x - width, metrics_df['Precision'], width, label='Precision', color='#3498db')
bars2 = ax.bar(x, metrics_df['Recall'], width, label='Recall', color='#2ecc71')
bars3 = ax.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#f39c12')
 
ax.set_xlabel('Severity Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance: Random Forest', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(severity_names)
ax.legend()
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)
 
# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)
 
plt.tight_layout()
plt.savefig('results/rf_per_class_performance.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: rf_per_class_performance.png")
plt.close()
 
# Save metrics to CSV
metrics_df.to_csv('results/rf_per_class_metrics.csv', index=False)
print("✓ Saved: rf_per_class_metrics.csv")
 

 

Creating ROC curves for best model (XGBoost Tuned)...
✓ Saved: rf_roc_curves_best_model.png

Creating per-class performance analysis...

Per-Class Performance (Random Forest):
  Severity  Precision   Recall  F1-Score  Support
Severity 1   0.107401 0.704142  0.186374    845.0
Severity 2   0.907834 0.664537  0.767362  69176.0
Severity 3   0.467638 0.638794  0.539978  16683.0
Severity 4   0.124888 0.543055  0.203074   2311.0

✓ Saved: rf_per_class_performance.png
✓ Saved: rf_per_class_metrics.csv


In [33]:
"""
SECTION 7: MODEL INTERPRETATION WITH SHAP
==========================================
This section explains WHAT the model learned and WHY it makes predictions.

SHAP (SHapley Additive exPlanations):
- Shows feature importance globally
- Explains individual predictions
- Reveals feature interactions
- Generates business insights

"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

print("="*80)
print("SECTION 7: MODEL INTERPRETATION WITH SHAP")
print("="*80)

# ============================================================================
# STEP 1: LOAD BEST MODEL AND DATA
# ============================================================================

# Load best model (XGBoost Tuned)
best_model = joblib.load('models/xgboost_tuned_model.pkl')
model_name = "XGBoost"

print(f"✓ Loaded: {model_name}")
print(f"✓ Test samples: {X_test_scaled.shape[0]:,}")
print(f"✓ Features: {X_test_scaled.shape[1]}\n")

# ============================================================================
# STEP 2: CREATE SHAP EXPLAINER
# ============================================================================
print("STEP 2: Creating SHAP explainer...")
print("(This may take 2-3 minutes for the full test set...)\n")

# Use a sample for faster computation (10,000 samples is sufficient)
np.random.seed(42)
sample_size = min(10000, len(X_test_scaled))
sample_idx = np.random.choice(len(X_test_scaled), sample_size, replace=False)
X_sample = X_test_scaled.iloc[sample_idx]

# Create TreeExplainer (optimized for tree-based models)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X_sample)

print(f"✓ SHAP values computed for {sample_size:,} samples")
print(f"✓ Shape: {shap_values.values.shape}\n")

# ============================================================================
# STEP 3: GLOBAL FEATURE IMPORTANCE
# ============================================================================

# Calculate mean absolute SHAP values for each feature
mean_shap = np.abs(shap_values.values).mean(axis=0)

# For multiclass, average across all classes
if len(mean_shap.shape) > 1:
    mean_shap = mean_shap.mean(axis=1)

# Create importance dataframe
importance_df = pd.DataFrame({
    'Feature': X_test_scaled.columns,
    'SHAP_Importance': mean_shap
}).sort_values('SHAP_Importance', ascending=False)

# ============================================================================
# VISUALIZATION 1: SHAP Summary Plot (Bar)
# ============================================================================
print("Creating Visualization 1: Feature Importance Bar Plot...")

fig, ax = plt.subplots(figsize=(10, 8))

# Top 15 features
top_features = importance_df.head(15)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_features)))

ax.barh(range(len(top_features)), top_features['SHAP_Importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value| (Average Impact on Output)', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Features by SHAP Importance', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('shap_importance_bar.png', dpi=300, bbox_inches='tight')
print("✓ Saved: shap_importance_bar.png\n")
plt.close()

# ============================================================================
# VISUALIZATION 2: SHAP Summary Plot (Beeswarm)
# ============================================================================
print("Creating Visualization 2: SHAP Beeswarm Plot...")
print("(Shows feature importance + impact direction)\n")

# For multiclass, show summary for Severity 3 (most interesting - serious accidents)
# Class 2 in XGBoost = Severity 3 in original labels
severity_3_idx = 2

fig, ax = plt.subplots(figsize=(10, 10))
shap.summary_plot(shap_values[:,:,severity_3_idx], X_sample, 
                  max_display=20, show=False)
plt.title('SHAP Summary: Severity 3 (Serious Accidents)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_beeswarm_severity3.png', dpi=300, bbox_inches='tight')
print("✓ Saved: shap_beeswarm_severity3.png")
plt.close()

# ============================================================================
# VISUALIZATION 3: SHAP Dependence Plots (Top 4 Features)
# ============================================================================
print("\nCreating Visualization 3: SHAP Dependence Plots...")
print("(Shows how each feature affects predictions)\n")

# Get top 4 features
top_4_features = importance_df.head(4)['Feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_4_features):
    feature_idx = list(X_test_scaled.columns).index(feature)
    
    # Create dependence plot for Severity 3
    shap.dependence_plot(
        feature_idx, 
        shap_values[:,:,severity_3_idx].values,
        X_sample, 
        ax=axes[idx],
        show=False
    )
    axes[idx].set_title(f'Impact of {feature}', fontsize=11, fontweight='bold')

plt.suptitle('SHAP Dependence Plots: Top 4 Features (Severity 3)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_dependence_plots.png', dpi=300, bbox_inches='tight')
print("✓ Saved: shap_dependence_plots.png\n")
plt.close()


# ============================================================================
# STEP 4: KEY INSIGHTS FROM SHAP ANALYSIS
# ============================================================================
insights = []

# Insight 1: Top driver features
top_5 = importance_df.head(5)['Feature'].tolist()
insights.append(f"1. TOP DRIVERS OF ACCIDENT SEVERITY:")
insights.append(f"   - {', '.join(top_5)}")

# Insight 2: Geographic vs temporal
geo_features = [f for f in top_5 if 'Lat' in f or 'Lng' in f or 'State' in f]
temporal_features = [f for f in top_5 if 'Hour' in f or 'Day' in f or 'Month' in f or 'Rush' in f]
weather_features = [f for f in top_5 if 'Temperature' in f or 'Weather' in f or 'Visibility' in f]

if geo_features:
    insights.append(f"\n2. LOCATION MATTERS:")
    insights.append(f"   - Geographic features ({', '.join(geo_features)}) in top 5")
    insights.append(f"   - Certain areas consistently have higher severity")

if temporal_features:
    insights.append(f"\n3. TIME PATTERNS:")
    insights.append(f"   - Temporal features ({', '.join(temporal_features)}) are important")
    insights.append(f"   - When accidents happen affects severity")

# Insight 3: Infrastructure impact
infra_features = [f for f in importance_df.head(20)['Feature'] 
                  if 'Traffic_Signal' in f or 'Junction' in f or 'Crossing' in f or 'Infrastructure' in f]
if infra_features:
    insights.append(f"\n4. INFRASTRUCTURE SAFETY:")
    insights.append(f"   - {', '.join(infra_features[:3])} impact severity")
    insights.append(f"   - Controlled intersections reduce severity")

# Insight 4: Weather influence
weather_in_top20 = importance_df.head(20)['Feature'].str.contains('Temperature|Weather|Visibility|Wind').sum()
if weather_in_top20 > 0:
    insights.append(f"\n5. WEATHER CONDITIONS:")
    insights.append(f"   - {weather_in_top20} weather features in top 20")
    insights.append(f"   - Confirms clear weather paradox from EDA")

# Print insights
print("\n" + "-"*80)
for insight in insights:
    print(insight)
print("-"*80)

# Save insights to file
with open('shap_key_insights.txt', 'w') as f:
    f.write("KEY INSIGHTS FROM SHAP ANALYSIS\n")
    f.write("="*80 + "\n\n")
    for insight in insights:
        f.write(insight + "\n")
    
    f.write("\n\nMODEL INTERPRETATION SUMMARY\n")
    f.write("="*80 + "\n")
    f.write(f"Model: {model_name}\n")

    f.write(f"Samples Analyzed: {sample_size:,}\n")
    f.write(f"Total Features: {X_test_scaled.shape[1]}\n")
    f.write(f"\nTop 5 Most Important Features:\n")
    for i, row in importance_df.head(5).iterrows():
        f.write(f"  {i+1}. {row['Feature']}: {row['SHAP_Importance']:.4f}\n")

print("\n✓ Saved: shap_key_insights.txt")




SECTION 7: MODEL INTERPRETATION WITH SHAP
✓ Loaded: XGBoost
✓ Test samples: 89,015
✓ Features: 78

STEP 2: Creating SHAP explainer...
(This may take 2-3 minutes for the full test set...)

✓ SHAP values computed for 10,000 samples
✓ Shape: (10000, 78, 4)

Creating Visualization 1: Feature Importance Bar Plot...
✓ Saved: shap_importance_bar.png

Creating Visualization 2: SHAP Beeswarm Plot...
(Shows feature importance + impact direction)

✓ Saved: shap_beeswarm_severity3.png

Creating Visualization 3: SHAP Dependence Plots...
(Shows how each feature affects predictions)

✓ Saved: shap_dependence_plots.png


--------------------------------------------------------------------------------
1. TOP DRIVERS OF ACCIDENT SEVERITY:
   - Distance(mi), Month, Season_Winter, Start_Lng, Start_Lat

2. LOCATION MATTERS:
   - Geographic features (Start_Lng, Start_Lat) in top 5
   - Certain areas consistently have higher severity

3. TIME PATTERNS:
   - Temporal features (Month) are important
   - When a